# use spark approach 1 (caused issue)

run in local

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# spark-master is the service_name not the docker contaienr name and we use it if we are in the cluster network (i.e., we run within container)
spark = (
    SparkSession.builder.appName("TestApp")
    .master("spark://localhost:7077")
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # .config("spark.driver.port", "55555") # “pin” the driver port with this command will cause issue 
    .getOrCreate()
)

sc = spark.sparkContext

In [4]:
# Get executor memory status
executor_status = sc._jsc.sc().getExecutorMemoryStatus()

# Convert Java Map to a Python dictionary
executor_status_dict = sc._gateway.jvm.scala.collection.JavaConversions.mapAsJavaMap(executor_status)

# Get the keys (nodes)
nodes = list(executor_status_dict.keys())
nodes

['host.docker.internal:58105', '172.20.0.5:32783', '172.20.0.4:37303']

In [3]:
data = [("Alice", 1), ("Bob", 2), ("Charlie", 3)]
df = spark.createDataFrame(data, ["Name", "ID"])
df.show()

+-------+---+
|   Name| ID|
+-------+---+
|  Alice|  1|
|    Bob|  2|
|Charlie|  3|
+-------+---+



In [25]:
sales = spark.read.option("header", "true").csv("/data/practice/sales.csv") # read from shared volume
prod = spark.read.option("header", "true").csv("/data/practice/products.csv") # read from shared volume
users = spark.read.option("header", "true").csv("/data/practice/users.csv") # read from shared volume
sales.show(2), prod.show(2), users.show(2)

+--------+----------+-------+--------+-----+----------+
|order_id|product_id|user_id|quantity|price| timestamp|
+--------+----------+-------+--------+-----+----------+
|       0|       553|   4397|       8|490.6|2023-08-18|
|       1|       441|   6066|       2|23.87|2023-10-09|
+--------+----------+-------+--------+-----+----------+
only showing top 2 rows

+----------+----------+--------+-----------+
|product_id|      name|category|supplier_id|
+----------+----------+--------+-----------+
|         1|rfzgjtsqqi|    Home|         50|
|         2|eqlmnfezzy|  Sports|         47|
+----------+----------+--------+-----------+
only showing top 2 rows

+-------+--------+---------+---+
|user_id|    name|  country|age|
+-------+--------+---------+---+
|      1|qivgzzqh|Australia| 24|
|      2|goghrglv|      USA| 30|
+-------+--------+---------+---+
only showing top 2 rows



(None, None, None)

In [26]:
sales.rdd.getNumPartitions()

1

In [31]:
sales_10p = sales.repartition(10)
sales_10p.rdd.getNumPartitions(), sales_10p.rdd.glom().map(len).collect()

(10, [10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000])

In [32]:
sales_4p = sales_10p.coalesce(4)
sales_4p.rdd.getNumPartitions(), sales_4p.rdd.glom().map(len).collect()

(4, [30000, 30000, 20000, 20000])

In [35]:
sales.filter(F.col("price") > 300).collect()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

# use spark approach 2

In [4]:
import pyspark.sql.functions as F

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("portforwarding").master("spark://spark-master:7077").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/26 20:08:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sales = spark.read.option("header", "true").csv("/data/practice/sales.csv") # read from shared volume
prod = spark.read.option("header", "true").csv("/data/practice/products.csv") # read from shared volume
users = spark.read.option("header", "true").csv("/data/practice/users.csv") # read from shared volume
sales.show(2), prod.show(2), users.show(2)

+--------+----------+-------+--------+-----+----------+
|order_id|product_id|user_id|quantity|price| timestamp|
+--------+----------+-------+--------+-----+----------+
|       0|       553|   4397|       8|490.6|2023-08-18|
|       1|       441|   6066|       2|23.87|2023-10-09|
+--------+----------+-------+--------+-----+----------+
only showing top 2 rows

+----------+----------+--------+-----------+
|product_id|      name|category|supplier_id|
+----------+----------+--------+-----------+
|         1|rfzgjtsqqi|    Home|         50|
|         2|eqlmnfezzy|  Sports|         47|
+----------+----------+--------+-----------+
only showing top 2 rows

+-------+--------+---------+---+
|user_id|    name|  country|age|
+-------+--------+---------+---+
|      1|qivgzzqh|Australia| 24|
|      2|goghrglv|      USA| 30|
+-------+--------+---------+---+
only showing top 2 rows



(None, None, None)

In [14]:
sales_10p = sales.repartition(10)
sales_4p = sales_10p.coalesce(4)

sales_4p.filter(F.col("price") > 300).select(["price", "timestamp"]).toPandas()

,price,timestamp
0,844.59,2023-02-09
1,667.66,2023-04-22
2,879.62,2023-04-12
3,369.5,2023-08-23
4,957.87,2023-04-11
...,...,...
70430,434.39,2023-09-03
70431,663.82,2023-11-07
70432,797.45,2023-12-28
70433,373.82,2023-09-20


In [16]:
sales.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [24]:
sales_4p.withColumn("total", F.col("price")*F.col("quantity")).sample(fraction=0.0001, seed=42).show()

+--------+----------+-------+--------+------+----------+------------------+
|order_id|product_id|user_id|quantity| price| timestamp|             total|
+--------+----------+-------+--------+------+----------+------------------+
|   14468|        96|   8757|       5|670.03|2023-08-14|3350.1499999999996|
|     516|       646|   5091|       7|329.57|2023-11-27|           2306.99|
|   13187|       590|   6022|       6| 778.3|2023-03-01| 4669.799999999999|
|   69640|       628|    146|       7| 43.25|2023-03-23|            302.75|
|   81261|       137|   3709|       1|379.97|2023-02-13|            379.97|
|   70007|       711|   6548|       4|966.44|2023-06-21|           3865.76|
+--------+----------+-------+--------+------+----------+------------------+



In [32]:
sales_4p.groupBy("product_id").agg(
    F.count("*").alias("shomar"),
    F.sum(F.col("quantity")).alias("total_Q"),
    F.round(F.avg(F.col("price")),2).alias("avg_P")
).show(2)

+----------+------+-------+------+
|product_id|shomar|total_Q| avg_P|
+----------+------+-------+------+
|       691|   117|  676.0|527.19|
|       467|    84|  502.0|476.77|
+----------+------+-------+------+
only showing top 2 rows



In [38]:
su = sales_4p.join(users.select(["user_id","name", "country"]), "user_id")
su.orderBy(F.desc("price")).show(2)

+-------+--------+----------+--------+------+----------+--------+-------+
|user_id|order_id|product_id|quantity| price| timestamp|    name|country|
+-------+--------+----------+--------+------+----------+--------+-------+
|   1438|   10866|       249|       9|999.98|2023-06-15|xbqdwfek| Canada|
|    812|   24923|       950|       5|999.96|2023-02-06|faanpqow|  Japan|
+-------+--------+----------+--------+------+----------+--------+-------+
only showing top 2 rows



In [43]:
sup = su.join(F.broadcast(prod), "product_id")
sup.show(2)

+----------+-------+--------+--------+------+----------+--------+---------+----------+--------+-----------+
|product_id|user_id|order_id|quantity| price| timestamp|    name|  country|      name|category|supplier_id|
+----------+-------+--------+--------+------+----------+--------+---------+----------+--------+-----------+
|       774|   2112|    6742|      10|959.22|2023-10-20|gttutrqx|  Germany|qeexcslmli|    Food|         40|
|       722|   2260|   94356|       1|827.16|2023-11-28|dapqrenh|Australia|qshkjjahsp|    Home|         31|
+----------+-------+--------+--------+------+----------+--------+---------+----------+--------+-----------+
only showing top 2 rows

